In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. Project setup
# --------------------------------------------------

PROJECT_ROOT = Path.cwd()

# If the notebook is being run from the notebooks folder,
# move one level up to the project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ANALYSIS_DIR = PROJECT_ROOT / "data" / "processed"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", ANALYSIS_DIR)

import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. Load and prepare movie data
# --------------------------------------------------

ANALYSIS_DIR = PROJECT_ROOT / "data" / "processed"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

movies = pd.read_csv(
    ANALYSIS_DIR / "movies.csv"
)

movies["release_year"] = pd.to_numeric(
    movies["release_year"], errors="coerce"
)

movies["budget"] = pd.to_numeric(
    movies["budget"], errors="coerce"
)

movies["revenue"] = pd.to_numeric(
    movies["revenue"], errors="coerce"
)

# Do not silently treat missing or invalid financial values as zero.
movies["profit"] = movies["revenue"] - movies["budget"]

movies["movie_roi_pct"] = np.where(
    movies["budget"] > 0,
    (movies["profit"] / movies["budget"]) * 100,
    np.nan
)

movies["revenue_multiple"] = np.where(
    movies["budget"] > 0,
    movies["revenue"] / movies["budget"],
    np.nan
)

# --------------------------------------------------
# 2. Identify actual multi-film bundles
# --------------------------------------------------

collection_sizes = (
    movies.dropna(subset=["collection_id"])
    .groupby("collection_id")["tmdb_id"]
    .nunique()
)

movies["collection_candidate_count"] = (
    movies["collection_id"].map(collection_sizes)
)

movies["licensing_type"] = np.select(
    [
        movies["collection_id"].isna(),
        movies["collection_candidate_count"].ge(2)
    ],
    [
        "Standalone film",
        "Franchise bundle"
    ],
    default="Single-film collection candidate"
)

print("LICENSING UNIT CLASSIFICATION")
print(movies["licensing_type"].value_counts())

# --------------------------------------------------
# 3. Create the two required analysis periods
# --------------------------------------------------

periods = {
    "All release years": movies.copy(),
    "2000 onward": movies[
        movies["release_year"] >= 2000
    ].copy()
}

unit_datasets = {}

for period_name, period_movies in periods.items():

    # Exclude collection members without another candidate
    # from the direct bundle-versus-standalone comparison.
    comparable = period_movies[
        period_movies["licensing_type"].isin(
            ["Franchise bundle", "Standalone film"]
        )
    ].copy()

    # A franchise is one collection; a standalone film is
    # its own individual licensing unit.
    comparable["licensing_unit_id"] = np.where(
        comparable["licensing_type"] == "Franchise bundle",
        "collection_" + comparable["collection_id"].astype(str),
        "movie_" + comparable["tmdb_id"].astype(str)
    )

    units = (
        comparable.groupby(
            ["licensing_type", "licensing_unit_id"],
            as_index=False
        )
        .agg(
            titles=("title", lambda x: " | ".join(x)),
            film_count=("tmdb_id", "nunique"),
            total_budget=("budget", "sum"),
            total_revenue=("revenue", "sum"),
            total_profit=("profit", "sum"),
            median_film_roi_pct=("movie_roi_pct", "median")
        )
    )

    units["portfolio_roi_pct"] = np.where(
        units["total_budget"] > 0,
        units["total_profit"] / units["total_budget"] * 100,
        np.nan
    )

    units["revenue_multiple"] = np.where(
        units["total_budget"] > 0,
        units["total_revenue"] / units["total_budget"],
        np.nan
    )

    unit_datasets[period_name] = units

# --------------------------------------------------
# 4. Compare franchise bundles and standalone films
# --------------------------------------------------

comparison_results = []

for period_name, units in unit_datasets.items():

    for licensing_type, group in units.groupby("licensing_type"):

        total_budget = group["total_budget"].sum()
        total_revenue = group["total_revenue"].sum()
        total_profit = group["total_profit"].sum()

        comparison_results.append({
            "period": period_name,
            "licensing_type": licensing_type,
            "licensing_units": len(group),
            "films": group["film_count"].sum(),
            "total_budget": total_budget,
            "total_revenue": total_revenue,
            "total_profit": total_profit,
            "weighted_portfolio_roi_pct": (
                total_profit / total_budget * 100
                if total_budget > 0 else np.nan
            ),
            "median_unit_roi_pct": group[
                "portfolio_roi_pct"
            ].median(),
            "median_revenue_multiple": group[
                "revenue_multiple"
            ].median()
        })

comparison_df = pd.DataFrame(comparison_results)

print("\nFRANCHISE VS STANDALONE COMPARISON")
print("=" * 80)

print(
    comparison_df.round(2).to_string(index=False)
)

# --------------------------------------------------
# 5. Save analysis outputs
# --------------------------------------------------

movies.to_csv(
    ANALYSIS_DIR / "movie_financial_metrics.csv",
    index=False
)

comparison_df.to_csv(
    ANALYSIS_DIR / "licensing_comparison.csv",
    index=False
)

for period_name, units in unit_datasets.items():
    file_name = (
        "licensing_units_all_years.csv"
        if period_name == "All release years"
        else "licensing_units_2000_onward.csv"
    )

    units.to_csv(
        ANALYSIS_DIR / file_name,
        index=False
    )

print("\nANALYSIS OUTPUTS SAVED")
print("movie_financial_metrics.csv")
print("licensing_comparison.csv")
print("licensing_units_all_years.csv")
print("licensing_units_2000_onward.csv")

# --------------------------------------------------
# 6. Basic integrity checks
# --------------------------------------------------

assert movies["tmdb_id"].is_unique
assert comparison_df["period"].nunique() == 2
assert set(comparison_df["licensing_type"]) == {
    "Franchise bundle",
    "Standalone film"
}

print("\nAll financial-analysis integrity checks passed.")


Matplotlib is building the font cache; this may take a moment.


Project root: c:\Users\SuraiyaMohammed\Desktop\StreamVault_Licensing_Analysis
Processed data directory: c:\Users\SuraiyaMohammed\Desktop\StreamVault_Licensing_Analysis\data\processed
LICENSING UNIT CLASSIFICATION
licensing_type
Franchise bundle                    87
Standalone film                     16
Single-film collection candidate     9
Name: count, dtype: int64

FRANCHISE VS STANDALONE COMPARISON
           period   licensing_type  licensing_units  films  total_budget  total_revenue  total_profit  weighted_portfolio_roi_pct  median_unit_roi_pct  median_revenue_multiple
All release years Franchise bundle               23     87   10643250000    58768179240   48124929240                      452.16               455.11                     5.55
All release years  Standalone film               16     16     891700000     4324338741    3432638741                      384.95               284.48                     3.84
      2000 onward Franchise bundle               20     59    935